# LifeMate fine-tune ASR (Colab)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Option A: unzip the code package uploaded to Drive (zip from scripts/pack_code.sh).
!mkdir -p /content/lifemate && unzip -oq "$(ls -t /content/drive/MyDrive/zac2026/lifemate-code-*.zip | head -1)" -d /content/lifemate
# Option B: !git clone <repo-url> /content/lifemate
%cd /content/lifemate

In [ ]:
%pip install -q -r requirements/notebook.txt -r requirements/train.txt pytest
# vllm is only needed for the agent-eval stage (llm pipeline "eval"); pinned from uv.lock (0.32.0).
!nvidia-smi > /dev/null 2>&1 && pip install -q vllm==0.32.0 || echo "no GPU: skipping vllm"

In [ ]:
!python -c "from pathlib import Path; from lifemate.envinfo import detect; print(detect(Path('.')))"

In [ ]:
!python -m pytest -q tests/train -m "not data and not tokenizer"

In [ ]:
# Drive is the work dir (/content/drive/MyDrive/zac2026/runs/train/<name>), so a second session resumes by itself.
!ls -la /content/drive/MyDrive/zac2026/runs/train 2>/dev/null || echo "no previous training on Drive"

In [ ]:
# One command (SPEC 6.4.5): data -> train -> select -> merge -> measure.
# Only run after the ASR measurements say so (SPEC 6.4, approach 5.5.2): gold-vs-ASR score gap >= 3 or
# entity miss rate >= 10% after LLM v2.
!python -m lifemate.train.asr_pipeline --config configs/train/asr_v1.toml --env colab

In [ ]:
!find /content/drive/MyDrive/zac2026/runs/train -maxdepth 3 -name "metrics.jsonl" -o -maxdepth 3 -name "measure.json" | head

In [ ]:
# Nothing to do: the work dir is on Drive. Copy the merged model to models/<name>/ when registering it.